<a href="https://colab.research.google.com/github/dheeraj-lab/RAG/blob/main/rag_based_qa_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:
!pip install -q sentence-transformers transformers

In [37]:
from sentence_transformers import SentenceTransformer, util
embedder = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [38]:
raw_text = """
Backpropagation is the method that computes gradients in a neural network. It starts at the loss and moves backward through the network, finding the gradient of the loss with respect to every weight and bias. It only computes gradients. It does not update the weights.

The chain rule is the core of backpropagation. The gradient of a node equals the gradient coming from above multiplied by the local derivative of that node. If one node is used in two places, the gradients from both paths are added together.

The gradient of a weight equals delta multiplied by the input of that weight, where delta is the derivative of the loss with respect to the neuron's z value. The gradient of a bias equals delta, because the input of a bias is always 1. Every weight and bias gets its own separate gradient.

The weight update rule is w_new = w_old - lr * gradient. The optimizer performs this update, not backpropagation. The minus sign moves the weight opposite to the gradient so the loss goes down. The learning rate lr controls how big each step is.

If the learning rate is too big, the loss jumps around or diverges. If the learning rate is too small, training is very slow. The loss can only go down reliably when the learning rate is small enough.

Mean squared error (MSE) is a loss for regression and equals the average of (y - y_hat) squared. Binary cross-entropy is the loss for two-class classification and is used with a sigmoid output. Categorical cross-entropy is the loss for many classes and is used with softmax. Language models like GPT also train with cross-entropy on next-token prediction.

For sigmoid with binary cross-entropy, the gradient of the loss with respect to z is simply y_hat - y. This simple form is why sigmoid and binary cross-entropy are used together.

Autograd is PyTorch's automatic differentiation. Calling loss.backward() computes the gradient of every parameter and stores it in the .grad field. It does not update the weights. The optimizer.step() call updates the weights using those gradients.

PyTorch accumulates gradients, meaning new gradients are added to old ones. This is why optimizer.zero_grad() is needed in every training step. Without zero_grad, gradients from earlier batches pile up and the updates become wrong.

The values from the forward pass are stored because the backward pass needs them to compute local derivatives. This is why training uses much more memory than just running the model.

A vanishing gradient happens when the gradient becomes tiny as it moves backward through many layers. Each layer multiplies the gradient by the weight times the activation derivative. If this factor is less than 1, repeated multiplication shrinks the gradient toward zero, and the early layers stop learning. The sigmoid derivative is at most 0.25, so sigmoid makes vanishing gradients worse.

An exploding gradient happens when the factor per layer is greater than 1, so the gradient grows very large. The weights can become NaN or the loss can jump wildly. The main fix is gradient clipping, which limits the size of the gradient.

Fixes for vanishing gradients are using ReLU or GELU activations, good weight initialization such as Xavier or He, BatchNorm or LayerNorm, and residual skip connections. Transformers use skip connections and LayerNorm for this reason.

SGD is stochastic gradient descent. It updates weights with w = w - lr * gradient using one learning rate for all parameters, and it can be noisy. Momentum remembers the average direction of past gradients to smooth the updates.

Adam combines momentum with a separate adaptive step size for each parameter. A parameter with large gradients gets smaller steps and a parameter with small gradients gets bigger steps. Adam, and its variant AdamW, is the default optimizer for training transformers and large language models.

A token embedding is a vector, a long list of numbers, that represents one token. Embeddings start random and are learned by backpropagation during training. A single number inside the vector has no meaning alone. The meaning lies in the whole vector and its directions.

Words that appear in similar contexts end up with similar vectors. This happens as a side effect of training on next-word prediction, not because anyone programmed grammar rules. This is why words like blue, white and big end up close together.

A real vector can have thousands of dimensions, for example 12288 in GPT-3 and 384 in the all-MiniLM-L6-v2 model. We cannot visualize that many dimensions, but the math works the same as in 3D. A vector is just a list of numbers, and the dot product just multiplies and adds them.

Self-attention lets each token collect information from earlier tokens. Each token produces a Query (what am I looking for), a Key (what do I contain) and a Value (what information I will give). The score between tokens is the dot product of Query and Key. A causal mask hides future tokens, softmax turns the scores into weights that sum to 1, and the output is the weighted mix of Value vectors. The formula is softmax(Q K transpose / sqrt(d)) V.

Q, K and V come from learned weight matrices Wq, Wk and Wv. The Query-Key dot product becomes large for useful token pairs because training pushed Wq and Wk in that direction. When a token gets useful information from another token, the loss goes down, and backpropagation adjusts the matrices to increase that attention. Nobody hard-coded which words should attend to which.

Multi-head attention runs the attention process several times with different weights. Each head can learn a different kind of relationship, such as adjectives to nouns or verbs to subjects. The results of all heads are combined. The division by sqrt(d) keeps the dot products from getting too large for softmax.

A transformer block contains an attention layer followed by an MLP, plus skip connections and LayerNorm. Attention mixes information between tokens. The MLP processes each token on its own and is where much of the model's knowledge is thought to be stored. Large models stack 12 to 96 of these blocks.

An MLP, or multi-layer perceptron, is a fully connected neural network. Each layer computes z = W x + b and then applies an activation function. The activation is needed because without it, many linear layers would collapse into a single linear layer. In a transformer, the MLP expands the vector to 4 times its size, applies GELU, and shrinks it back.

Hugging Face is a company and website that hosts AI models, like GitHub for models. The transformers library is a Python library from Hugging Face for downloading and running models. GPT-2 is a model. A pipeline runs a model in a few lines of code.

GPT-2 is a decoder model that predicts the next token and generates text. DistilBERT is an encoder model that reads a whole sentence and outputs a label such as POSITIVE or NEGATIVE. Both are transformers but they are built for different tasks.

Sentiment analysis finds the feeling of a text, such as positive or negative. It is a classification task. The label is the category the model chooses and the score is how confident the model is. A high score does not prove the answer is correct.

The sentiment model has limits. It had no neutral label, so neutral text like okay, nothing special was forced into positive or negative. Sarcasm, such as Wow great phone, it broke in one day, fooled it. It handled negation like not bad and I don't hate it correctly, which shows attention at work.

A tokenizer splits text into tokens and maps each token to an ID. Common words are one token, while rare or new words are split into pieces, for example unbelievable becomes un, bel, iev, able. The tokenizer is case sensitive, so Hello and hello are different tokens. The symbol G with a dot above means a space came before the word.

GPT-2's tokenizer works on bytes and its vocabulary is mostly English. Hindi text needs many more tokens, for example a 6 character Hindi word took 12 tokens. More tokens means higher cost, slower responses and less room in the context window.

The full flow inside a language model is text, tokens, IDs, embedding vectors, attention, MLP, and finally a prediction of the next token. The model never sees words, only token IDs and their vectors.

Temperature controls how random text generation is. The probability is softmax of the scores divided by T. A low temperature like 0.3 makes the model repeat safe words and loop the same sentence. A high temperature like 1.5 flattens the probabilities, so the text becomes random and drifts off topic. Use low temperature for facts and RAG answers and higher temperature for creative writing.

A sentence embedding is one vector for a whole sentence. Inside the model, attention works on token vectors, and then mean pooling averages the token vectors into one sentence vector. Sentences of any length become the same size vector, for example 384 numbers. Sentence embeddings are used to compare sentences for search.

Cosine similarity measures how close the directions of two vectors are. A value of 1 means the same direction and similar meaning, 0 means unrelated, and negative values are noise or opposite directions. Two sentences with similar meaning but different words, like a cat sits on the mat and a kitten rests on the rug, still get a high score.

Semantic search finds documents by meaning instead of exact words. Document vectors are created once with the same model used for the query. The query vector is compared to every document vector using cosine similarity, and the top results are returned. Search always returns something, even when nothing is relevant, so a score threshold is needed.

RAG stands for retrieval augmented generation. It has three parts. Retrieve means find the chunks of text closest to the question. Augment means put those chunks into the prompt as context. Generate means the language model answers using only that context. If no chunk passes the score threshold, the system says it does not know instead of guessing.

Chunking means splitting documents into small pieces of a few lines before embedding them. Whole documents as one vector give poor search results because the meaning gets mixed. A system message tells the model to answer only from the given context. A small language model can still give wrong answers even with the right context.
"""

chunks = [c.strip() for c in raw_text.split("\n\n") if c.strip()]
print(len(chunks), "chunks")

36 chunks


In [39]:
#make the chunks vector
chunk_emb = embedder.encode(chunks)

chunk_emb.shape

(36, 384)

In [40]:
#retrieve function
def retrieve(question, k=2,threshold=0.3):
  q_emb = embedder.encode(question)
  hits = util.semantic_search(q_emb,chunk_emb,top_k=k)[0]
  hits =[h for h in hits if h["score"]>= threshold]
  return [(round(h["score"],3), chunks[h["corpus_id"]]) for h in hits]





In [41]:
from transformers import pipeline
llm = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [42]:
#ask function
def ask(question):
  results= retrieve(question)
  if not results:
    return "Answer does not exist."

  context = "\n\n".join(text for _, text in results)
  messages = [
      {"role": "system", "content": "Answer ONLY using the context given. If the answer is not in the context, say you don't know. Keep the answer short."},
      {"role": "system", "content": f"Context:\n{context}\n\nQuestion:{question}"},
  ]

  out = llm(messages, max_new_tokens=120, do_sample=False)
  return out[0]["generated_text"][-1]["content"]

print(ask("What does backpropagation do?"))
print(ask("Why is Hindi expensive in GPT-2?"))
print(ask("Who won the cricket world cup?"))

computes gradients
Hindi text requires many more tokens than English text, which makes it more expensive to generate with GPT-2.
Answer does not exist.
